# Distant Contradiction Auditor - Legal-BERT Training on Kaggle

Use a Kaggle GPU notebook. Enable **Settings > Accelerator > GPU** and **Internet** so the pretrained Legal-BERT weights can be downloaded. Run the cells in order.

Dataset: `/kaggle/input/datasets/madhumangayi/dca-dataset/pairs_colab_full.jsonl`
Output: `/kaggle/working/`

In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn

from pathlib import Path
import torch

DATASET_DIR = Path('/kaggle/input/datasets/madhumangayi/dca-dataset')
PAIRS = DATASET_DIR / 'pairs_colab_full.jsonl'
OUT = Path('/kaggle/working/legalbert-dca')
CHECKPOINTS = Path('/kaggle/working/dca-checkpoints')

if not PAIRS.exists():
    raise FileNotFoundError(f'Dataset not found: {PAIRS}')

OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dataset:', PAIRS)
print('Dataset size:', f'{PAIRS.stat().st_size:,}', 'bytes')
print('Output:', OUT)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL = 'nlpaueb/legal-bert-base-uncased'
MAX_LEN = 256

rows = [json.loads(line) for line in PAIRS.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip()]
by = {}
for row in rows:
    split_name = row.get('split', 'train')
    by.setdefault(split_name, []).append({
        'text_a': row['text_a'][:1200],
        'text_b': row['text_b'][:1200],
        'label': int(row['label']),
        'gap_bin': row.get('gap_bin', 'unknown'),
        'type': row.get('type', 'Inconsistency'),
    })
ds = DatasetDict({name: Dataset.from_list(values) for name, values in by.items()})
print('Dataset splits:', ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LEN, padding=False)

encoded = ds.map(tokenize, batched=True, remove_columns=['text_a', 'text_b'])
encoded = encoded.rename_column('label', 'labels')
print('Encoded splits:', encoded)

In [ ]:
import inspect
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    return {'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}

use_cuda = torch.cuda.is_available()
train_batch_size = 16 if use_cuda else 2
eval_batch_size = 32 if use_cuda else 4
training_parameters = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs = {
    'output_dir': str(CHECKPOINTS),
    'per_device_train_batch_size': train_batch_size,
    'per_device_eval_batch_size': eval_batch_size,
    'gradient_accumulation_steps': 2,
    'num_train_epochs': 6,
    'learning_rate': 2e-5,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'fp16': use_cuda,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'greater_is_better': True,
    'save_total_limit': 2,
    'logging_steps': 50,
    'dataloader_num_workers': 2,
    'report_to': [],
    'seed': 13,
}

if 'warmup_ratio' in training_parameters:
    training_kwargs['warmup_ratio'] = 0.1
else:
    steps_per_epoch = max(1, len(encoded['train']) // (train_batch_size * 2))
    training_kwargs['warmup_steps'] = max(1, int(steps_per_epoch * 6 * 0.1))

if 'eval_strategy' in training_parameters:
    training_kwargs['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in training_parameters:
    training_kwargs['evaluation_strategy'] = 'epoch'

args = TrainingArguments(**training_kwargs)
eval_ds = encoded['dev'] if 'dev' in encoded else encoded['test']
trainer_parameters = inspect.signature(Trainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': encoded['train'],
    'eval_dataset': eval_ds,
    'compute_metrics': compute_metrics,
    'callbacks': [EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
}
if 'processing_class' in trainer_parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)
print(f'Starting training on {DEVICE} with batch size {train_batch_size}...')
trainer.train()
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
print('Saved model to:', OUT)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

test_result = trainer.predict(encoded['test'])
logits, labels = test_result.predictions, test_result.label_ids
predictions = np.argmax(logits, axis=-1)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, predictions, average='binary', zero_division=0
)
metrics = {
    'model': MODEL,
    'device': DEVICE,
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1': float(f1),
    'n_test': len(by['test']),
}
metrics_path = Path('/kaggle/working/metrics_legalbert.json')
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))
print('Metrics saved to:', metrics_path)

In [ ]:
import shutil

archive = shutil.make_archive('/kaggle/working/legalbert-dca', 'zip', OUT)
print('Model archive:', archive)
print('Download it from the Kaggle notebook output files.')